# Этап C **без присмотра**: запустить и уйти

> Парная к [`kaggle_phase_c.ipynb`](kaggle_phase_c.ipynb), которая
> для работы руками. Здесь замера скорости нет — он сделан, 1,95 вопр./с;
> проба встроена внутрь, потому что снаружи её запускать некому.

Эта тетрадь рассчитана **только** на **Save Version → Save & Run All
(Commit)**. Её не надо прогонять руками по ячейкам: commit стартует
с чистого листа, в новой сессии, и всё, что нужно, делает сам.

**Перед запуском, один раз:**

1. **+ Add Input** → датасеты `ru-fin-retrieval` и `ru-fin-queries`;
2. **Settings → Accelerator → GPU T4 ×2**, интернет включён;
3. **Add-ons → Secrets** → галочка `HF_TOKEN` (в новой тетради — заново);
4. **Save Version → Save & Run All (Commit)** и можно уходить.

Около четырёх часов: полчаса на установку и веса, 1,7 на 12 000 вопросов,
1,3 на фильтр учителя, полчаса на тест другой моделью. Предел сессии
Kaggle — двенадцать часов, запас большой.

Результат сохраняется сам: у пакетного прогона `/kaggle/working`
становится выводом версии, и забрать его можно через API. Скачивать
руками ничего не надо — в этом и смысл.

**Чего здесь нет нарочно:** замера скорости (он сделан, 1,95 вопр./с)
и пробного прогона на ста актах (тоже сделан). Повторять их — значит
тратить квоту на уже известное.

In [ ]:
# ──────── НАСТРОЙКИ ПРОГОНА ────────
# Меняются только здесь. Остальные ячейки читают отсюда, чтобы одна тетрадь
# годилась и на полный прогон, и на добор отдельных кусков: гонять генерацию
# обучающей выборки ради перезаписи теста — два часа видеокарты впустую.

TRAIN = 12000      # 0 — не трогать обучающую выборку
DEV = 300
TEST = 400

# Речевой режим выбирается на пачку целиком, поэтому пачка задаёт, сколько
# режимов успеет появиться. При 256 и цели 300 отрабатывают две пачки — и
# третьего режима, короткого поискового запроса, в наборе не оказывается
# вовсе. Для обучающей выборки это неважно (пачек много), для dev и теста
# решает: ученик обучался бы на коротких запросах и ни разу на них
# не проверялся.
BATCH = 64

ФИЛЬТР_ГЛУБИНА = 30   # порог скрипт назначит сам, половиной глубины
ПРОБА = True          # на доборе отдельных кусков выключается: путь проверен

ВЕСА = []
if TRAIN:
    ВЕСА.append(("Qwen/Qwen2.5-7B-Instruct-AWQ", 1800))
if DEV or TEST:
    ВЕСА.append(("NiGuLa/Vikhr-Nemo-12B-Instruct-R-21-09-24-awq-4bit", 2400))
ВЕСА.append(("BAAI/bge-reranker-v2-m3", 1200))

print(f"train={TRAIN} dev={DEV} test={TEST} пачка={BATCH} проба={ПРОБА}")
print("качаем:", ", ".join(и for и, _ in ВЕСА))

## 1. Карта и токен

Падать надо здесь, на первой секунде, а не через сорок минут на загрузке
весов. Без токена скачивание режется по скорости и может встать намертво,
а зависание не ошибка — его не ловит ни один `try`.

In [ ]:
import os, subprocess, sys

!nvidia-smi --query-gpu=name,memory.total --format=csv

# Токен необязателен, и это сознательно. Секреты Kaggle прицепляются только
# мышкой: у API поля для них нет вовсе, поэтому ядро, созданное командой,
# токена не увидит никогда. Раньше это означало «такие ядра запускать нельзя».
# Но 30.09 два прогона через API — этап B без ускорителя и реранкер — скачали
# веса без токена и досчитали. Значит запрет был слишком широким: токен
# ускоряет скачивание, но не является его условием.
# Страхует не токен, а предохранитель на скачивание в следующей ячейке.
ТОКЕН = False
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token and len(token) > 10:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = token
        ТОКЕН = True
        print("токен HuggingFace на месте, длина", len(token))
except Exception as e:
    print("токена нет:", str(e)[:120])
if not ТОКЕН:
    print("идём без токена: скачивание медленнее, но под предохранителем")

# Карта проверяется подпроцессом, а НЕ через `import torch` здесь: установка
# vLLM следующей ячейкой обновляет torch и тянет библиотеки CUDA 13, а уже
# загруженный в этот процесс старый torch держит CUDA 12 из образа Kaggle.
# Дальше `import vllm` ищет libcudart.so.13 и не находит — прогон 29.09
# на этом и умер.
видно = subprocess.run([sys.executable, "-c",
                        "import torch; print(torch.cuda.get_device_name(0))"],
                       capture_output=True, text=True)
assert видно.returncode == 0, f"карта не видна: {видно.stderr[-400:]}"
print("карта:", видно.stdout.strip())

## 2. Библиотеки

vLLM тянет свою сборку torch с CUDA — два-четыре гигабайта. Пять-пятнадцать
минут норма. Вывод не глушится: отличить скачивание от перебора версий
зависимостей можно только по нему, а перебор крутится часами, и всё это
время квота идёт.

In [ ]:
import subprocess, sys, time

t = time.time()
for пакет in ("vllm", "sentence-transformers"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", пакет,
                    "--progress-bar", "off"], check=True)
print(f"установка заняла {(time.time() - t) / 60:.1f} мин")

# Проверка тоже подпроцессом, и это не педантизм: в процессе тетради
# уже живёт torch, подгруженный до установки, и `import vllm` здесь
# падает на libcudart.so.13, хотя в свежем процессе проходит.
# Тридцать секунд сейчас против четырёх часов, потраченных зря.
# Проверяются обе библиотеки: vLLM генерирует, sentence-transformers
# поднимает кросс-энкодер-учителя для третьего фильтра. Учитель нужен
# через два часа после старта, и узнавать тогда, что он не импортируется,
# поздно.
код = ("import vllm, sentence_transformers as st; "
       "print(vllm.__version__, st.__version__)")
проба = subprocess.run([sys.executable, "-c", код], capture_output=True, text=True)
assert проба.returncode == 0, f"библиотеки не импортируются:\n{проба.stderr[-1500:]}"
print("vllm и sentence-transformers:", проба.stdout.strip())

### 2б. Веса — заранее и под предохранителем

Скачиваются до всякой генерации, по одной модели, каждая со своим
пределом времени. Зависшее скачивание — не ошибка: его не ловит ни один
`try`, сессия просто стоит, пока не кончится квота, и однажды это уже
стоило проекту двух часов. Подпроцесс с `timeout` решает грубо и надёжно.

С токеном это две-три минуты, без токена — дольше, но предел не даст
уйти в тишину.


In [ ]:
import subprocess, sys, time

# Веса тянутся заранее и по одной модели, каждая со своим пределом времени.
# Зависшее скачивание — не ошибка: его не ловит ни один `try`, и сессия
# просто стоит, пока не кончится квота. Проект уже потерял на этом два часа.
# Подпроцесс с timeout решает это грубо и надёжно: не уложился — убит.
МОДЕЛИ = ВЕСА
код = ("import sys; from huggingface_hub import snapshot_download; "
       "p = snapshot_download(sys.argv[1], max_workers=8); print(p)")
for имя, предел in МОДЕЛИ:
    t = time.time()
    print(f"качаю {имя} (предел {предел // 60} мин)...", flush=True)
    try:
        r = subprocess.run([sys.executable, "-c", код, имя],
                           capture_output=True, text=True, timeout=предел)
    except subprocess.TimeoutExpired:
        raise SystemExit(f"{имя}: скачивание не уложилось в {предел // 60} мин. "
                         f"Прицепите HF_TOKEN мышкой либо загрузите веса "
                         f"на Kaggle отдельным датасетом.")
    assert r.returncode == 0, f"{имя} не скачалась:\n{r.stderr[-1200:]}"
    print(f"   готово за {(time.time() - t) / 60:.1f} мин", flush=True)

print("\nвсе веса на месте — дальше HuggingFace не нужен")

## 3. Код, корпус и каноническая нарезка

Код берётся из датасета, а не из этой тетради. Источники ранжируются:
вывод прежнего ядра тоже содержит копию кода, и если взять оттуда,
приедет старая версия — молча.

**Нарезка не пересчитывается, а берётся готовая.** Она зависит от версии
токенизатора, а установка vLLM тянет свой `transformers` и сдвигает
границы: 61 922 фрагмента вместо 62 594 на том же корпусе. Идентификатор
фрагмента позиционный (`акт#номер`), поэтому при сдвиге он не исчезает,
а начинает указывать на другой текст — эталон съехал бы молча, и выборка
на 12 000 примеров учила бы модель находить не то.

In [ ]:
import glob, gzip, os, shutil


def find_source(root="/kaggle/input", max_depth=4):
    found = []
    for cur, dirs, files in os.walk(root):
        if cur.count("/") - root.count("/") >= max_depth:
            dirs.clear()
        if "rufin" in dirs:
            stale = ("__notebook__.ipynb" in files or "embeddings" in dirs
                     or "__output__.json" in files)
            found.append((1 if stale else 0, cur))
    found.sort()
    return found[0][1] if found else None


src = find_source()
assert src, "датасет с кодом не подключён"
print("источник кода:", src)
for item in glob.glob(os.path.join(src, "*")):
    цель = os.path.join("/kaggle/working", os.path.basename(item))
    if os.path.isdir(item):
        shutil.copytree(item, цель, dirs_exist_ok=True)
    else:
        shutil.copy2(item, цель)
os.chdir("/kaggle/working")

готовая = (glob.glob("/kaggle/input/**/chunks_base.jsonl", recursive=True)
           + glob.glob("/kaggle/input/**/chunks_base.jsonl.gz", recursive=True))
assert готовая, "нет chunks_base.jsonl: подключите датасет ru-fin-queries"
os.makedirs("/kaggle/working/chunks", exist_ok=True)
цель = "/kaggle/working/chunks/base.jsonl"
if os.path.exists(цель):
    os.remove(цель)
opener = gzip.open if готовая[0].endswith(".gz") else open
with opener(готовая[0], "rb") as fi, open(цель, "wb") as fo:
    shutil.copyfileobj(fi, fo, length=1 << 20)

n = sum(1 for _ in open(цель, encoding="utf-8"))
print("нарезка:", готовая[0], "->", n, "фрагментов")
assert n == 62594, f"ожидалось 62594, а не {n}: нарезка не каноническая"

for имя in ("run_phase_c.py", "gpu_gen.py", "gpu_judge.py", "acts.jsonl"):
    assert os.path.exists(имя), f"нет {имя}: датасет приехал неполным"
print("сплит:", glob.glob("/kaggle/input/**/split.json", recursive=True))

## 4. Пробный прогон. Двадцать вопросов на выборку

Смысл — поймать брак за минуты, а не через три с половиной часа. Проверяются
ровно те места, которые на этой карте ещё ни разу не работали: запись файлов,
три речевых режима и третий фильтр (здесь с глубиной 5 — код тот же, цена
никакая).

Он почти бесплатен: веса обеих моделей и кросс-энкодера-учителя скачиваются
здесь и остаются в кэше сессии, поэтому большой прогон поднимет их с диска.
То есть эти минуты не добавляются к общему времени, а вычитаются из него.

Если проверки ниже не сойдутся — тетрадь упадёт **здесь**, потратив минуты,
а не часы.

In [ ]:
if not ПРОБА:
    print('проба выключена настройками: путь уже проверен')
else:
    import json, subprocess, sys, time

    env = dict(os.environ, VLLM_WORKER_MULTIPROC_METHOD="spawn")
    t = time.time()
    p = subprocess.run([sys.executable, "run_phase_c.py", "--probe", "0",
                        "--train", "20", "--dev", "20", "--test", "20",
                        "--teacher-filter", "--filter-depth", "10",
                        # Стиль выбирается на пачку целиком. При пачке по умолчанию
                        # (256) все двадцать вопросов выйдут одного режима, и
                        # различимость режимов проверить будет нечем — проба
                        # 30.09 на этом и споткнулась.
                        "--batch", "8",
                        "--out", "/kaggle/working/proba"],
                       cwd="/kaggle/working", env=env)
    print(f"\nпроба заняла {(time.time() - t) / 60:.1f} мин, код возврата {p.returncode}")
    assert p.returncode == 0, "проба упала — большой прогон упал бы так же, но позже"

    стили = set()
    for имя in ("synthetic_train", "synthetic_dev", "synthetic_test"):
        путь = f"/kaggle/working/proba/queries/{имя}.jsonl"
        assert os.path.exists(путь), f"проба не создала {имя}.jsonl"
        строки = [json.loads(l) for l in open(путь, encoding="utf-8")]
        assert строки, f"{имя}.jsonl пустой: все вопросы отсеяны фильтрами"
        print(f"\n=== {имя}: {len(строки)} ===")
        for з in строки[:4]:
            print(f"   [{з.get('style', '?')}] {з['text'][:110]}")
        for з in строки:
            стили.add(з.get("style"))
            assert з.get("gold_chunk_id"), "у вопроса нет эталонного фрагмента"
            assert з["gold_chunk_id"].count("#") == 1, "идентификатор фрагмента не тот"

    print("\nречевые режимы:", sorted(стили))
    assert len(стили) >= 2, ("режимы неразличимы: генерация свелась к одному стилю. "
                             "При пачке 8 и двадцати вопросах их должно быть 2-3")

    # Один из режимов — короткий поисковый запрос, он пишется без знака вопроса.
    # Если знак вопроса стоит у всех, значит стиль не доехал до модели,
    # и обучающая выборка получится однобокой.
    все = [json.loads(l) for имя in ("synthetic_train", "synthetic_dev", "synthetic_test")
           for l in open(f"/kaggle/working/proba/queries/{имя}.jsonl", encoding="utf-8")]
    без_знака = sum(1 for з in все if not з["text"].rstrip().endswith("?"))
    print(f"без знака вопроса: {без_знака} из {len(все)}")
    if not без_знака:
        # Не отказ: фильтр знака вопроса не требует ни от одного режима,
        # и модель вправе поставить его даже в коротком запросе. Но если
        # его нет ни у одного вопроса из шестидесяти — это повод посмотреть
        # на выборку глазами, а не повод выбрасывать три часа счёта.
        print("ВНИМАНИЕ: знак вопроса у всех. Короткий поисковый режим, "
              "возможно, не доехал — проверьте выборку после прогона")

    # Язык проверяется отдельно и строго. Модель для теста берётся из другого
    # семейства нарочно, но другое семейство хуже держит русский: на пробе
    # 30.09 Mistral выдал 11 английских вопросов из 20 в тесте. Фильтр такие
    # отбрасывает — здесь проверяется, что он действительно включён.
    import re
    кир, лат = re.compile(r"[а-яёА-ЯЁ]"), re.compile(r"[a-zA-Z]")
    чужие = [з["text"] for з in все
             if len(лат.findall(з["text"])) >= len(кир.findall(з["text"]))
             or re.search(r"[іїєґІЇЄҐ]", з["text"])]
    print(f"не по-русски: {len(чужие)} из {len(все)}")
    for т in чужие[:3]:
        print("   ", т[:100])
    assert not чужие, "нерусские вопросы прошли фильтр — проверьте queryfilter.po_russki"

    отчёт = json.load(open("/kaggle/working/proba/report_phase_c.json", encoding="utf-8"))
    assert any("фильтр учителя" in k for k in отчёт), "третий фильтр не отработал"
    print("\nпроба сошлась, иду на полный прогон")

### Перед тяжёлой частью: сколько осталось места

Сессия с картой даёт меньше оперативной памяти, чем сессия без неё,
а третий фильтр строит индекс BM25 по всем 62 594 фрагментам. Если
прогон потом умрёт молча, эти три числа скажут, почему.


In [ ]:
import shutil, subprocess, sys

# Диагностика на случай, если прогон умрёт молча: сессия с картой даёт
# меньше оперативной памяти, чем сессия без неё, а третий фильтр строит
# индекс BM25 по 62 594 фрагментам. Если места мало, это видно здесь,
# а не по загадочному обрыву на третьем часу.
своб = shutil.disk_usage("/kaggle/working")
print(f"диск: свободно {своб.free / 2**30:.1f} ГБ из {своб.total / 2**30:.1f}")
память = subprocess.run([sys.executable, "-c",
                         "import psutil; m = psutil.virtual_memory(); "
                         "print(f'{m.available/2**30:.1f} {m.total/2**30:.1f}')"],
                        capture_output=True, text=True)
if память.returncode == 0:
    дост, всего = память.stdout.split()
    print(f"память: доступно {дост} ГБ из {всего}")
print(f"веса моделей в кэше: "
      f"{sum(f.stat().st_size for f in __import__('pathlib').Path('/root/.cache').rglob('*') if f.is_file()) / 2**30:.1f} ГБ")

## 5. Генерация. Около четырёх часов

Одним прогоном, потому что шаги делят загрузку корпуса и сами выгружают
модель перед следующей:

* **обучающая выборка** — цель 12 000, Qwen2.5-7B-Instruct-AWQ;
* **dev** — 300 вопросов **другой** моделью, Vikhr-Nemo-12B (это
  Mistral-Nemo, дообученный на русском);
* **тест** — 400 вопросов ею же. Другое семейство обязательно: иначе
  тест померит не качество поиска, а то, насколько ученик выучил стиль
  своего же генератора. Но семейство мало выбрать — оно должно ещё
  и держать русский, и на этом Mistral-7B и заменён;
* **третий фильтр** — учитель-кросс-энкодер, глубина 30. Порог скрипт
  назначает сам, половиной глубины: эталон обязан попасть в первые 15
  из 30. Жёсткий порог 50 при глубине 30 пропускал бы всё подряд —
  эталон всегда среди кандидатов, и место у него не хуже тридцатого.

**Чего ждать по числам.** Пригодных как источник фрагментов в обучающей
части 16 431 из 44 358 — то есть запас над целью в 12 000 небольшой.
Два первых фильтра отсеивают часть, третий отсеивает ещё; итог, скорее
всего, окажется между восемью и десятью тысячами. Это нормально и
предусмотрено: разница между 8 000 и 12 000 примеров на дообучении
маленькой модели меньше, чем разница между «обучение состоялось»
и «квота кончилась в середине». Если вопросы кончатся раньше цели,
`make_queries` просто остановится — это не падение.

**Время.** Замер дал 1,95 вопроса в секунду до фильтров. Двенадцать тысяч
принятых — это около четырнадцати тысяч попыток, то есть 2–2,3 часа,
плюс час с небольшим на третий фильтр, плюс dev и тест. Итого около
четырёх часов; предел сессии Kaggle много больше.

`VLLM_WORKER_MULTIPROC_METHOD=spawn` обязателен: vLLM поднимает движок
отдельным процессом и по умолчанию форкается, а форк не годится, когда
CUDA уже тронута в родителе.

Шаги независимы и проверяют, нет ли готового файла, так что обрыв
не уносит сделанное. Фильтр переписывает файл только после того, как
досчитает: упади он — на диске останется нефильтрованная выборка,
а не пустота.

In [ ]:
import subprocess, sys, time

env = dict(os.environ, VLLM_WORKER_MULTIPROC_METHOD="spawn")
cmd = [sys.executable, "run_phase_c.py", "--probe", "0",
       "--train", str(TRAIN), "--dev", str(DEV), "--test", str(TEST),
       "--batch", str(BATCH),
       "--teacher-filter", "--filter-depth", str(ФИЛЬТР_ГЛУБИНА),
       "--out", "/kaggle/working/gen"]
print("запуск:", " ".join(cmd), flush=True)

t = time.time()
p = subprocess.run(cmd, cwd="/kaggle/working", env=env)

# Код возврата запоминается, но тетрадь здесь не роняется: прогон идёт без
# присмотра, и при падении на последнем шаге всё уже сгенерированное лежит
# на диске. Если оборвать ячейки сейчас, сводка не напечатается, а уборка
# не отработает — и в выводе версии окажется четверть гигабайта нарезки.
# Поэтому проверка кода возврата отложена в самый конец.
КОД_ВОЗВРАТА = p.returncode
print(f"\nгенерация заняла {(time.time() - t) / 3600:.2f} ч, "
      f"код возврата {КОД_ВОЗВРАТА}")
if КОД_ВОЗВРАТА:
    print("ПРОГОН УПАЛ. Ниже видно, какие шаги всё же успели записаться.")

## 6. Что получилось

Печатается для лога версии: по нему видно, сколько вопросов пережило
каждый фильтр, не открывая файлы.

In [ ]:
import json

путь_отчёта = "/kaggle/working/gen/report_phase_c.json"
if os.path.exists(путь_отчёта):
    r = json.load(open(путь_отчёта, encoding="utf-8"))
    for шаг, v in r.items():
        print(f"--- {шаг}")
        print("   ", v if not isinstance(v, dict)
              else {k: x for k, x in v.items() if not isinstance(x, dict)})
else:
    print("отчёта нет: прогон не дошёл до первого сохранения")

for имя in ("synthetic_train", "synthetic_dev", "synthetic_test"):
    путь = f"/kaggle/working/gen/queries/{имя}.jsonl"
    if not os.path.exists(путь):
        print(f"\n{имя}: файла нет"); continue
    строки = [json.loads(l) for l in open(путь, encoding="utf-8")]
    print(f"\n=== {имя}: {len(строки)} ===")
    for з in строки[:4]:
        print(f"   [{з.get('style', '?')}] {з['text'][:110]}")

## 7. Убрать тяжёлое из вывода

Нарезка и корпус приехали из входов и лежат в рабочей папке, а она целиком
становится выводом версии. Оставлять их — значит возить четверть гигабайта
туда и обратно без нужды. Всё ценное маленькое.

In [ ]:
import os, shutil

shutil.rmtree("/kaggle/working/chunks", ignore_errors=True)
shutil.rmtree("/kaggle/working/proba", ignore_errors=True)
for имя in ("acts.jsonl", "acts.jsonl.gz"):
    if os.path.exists(имя):
        os.remove(имя)

осталось = 0
for база, _, файлы in os.walk("/kaggle/working"):
    for f in файлы:
        осталось += os.path.getsize(os.path.join(база, f))
print(f"в выводе версии останется {осталось / 1048576:.1f} МБ\n")
for база, _, файлы in os.walk("/kaggle/working/gen"):
    for f in sorted(файлы):
        путь = os.path.join(база, f)
        print(f"   {os.path.relpath(путь, '/kaggle/working'):<44} "
              f"{os.path.getsize(путь) // 1024:>7} КБ")

# Теперь, когда всё сохранено и убрано,версия честно помечается неудачной,
# если прогон упал. Иначе в истории ядер останется зелёная галочка на прогоне,
# который ничего не досчитал, и это хуже, чем красный крест.
assert КОД_ВОЗВРАТА == 0, f"генерация вернула {КОД_ВОЗВРАТА}, смотрите вывод выше"